## Kiến trúc hệ thống:
1. **Detector Agent**: Phát hiện có manipulation hay không
2. **Analyzer Agent**: Xác định manipulator và kỹ thuật
3. **Evidence Agent**: Xác thực bằng chứng
4. **Meta Agent**: Tổng hợp kết quả cuối cùng

## Model:
- Base: Mistral-7B-Instruct-v0.3 (4-bit quantized)
- Fine-tuned: Custom adapter từ stage 1

## 1. Cài đặt thư viện

Cài đặt các thư viện cần thiết cho multi-agent system và model inference.

In [1]:
!pip install -U transformers peft accelerate langgraph bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Import thư viện

In [2]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import json 
import os
import pandas as pd 
import torch 
import torch.nn as nn
import numpy as np
from transformers import AutoModel, AutoTokenizer
from typing import Dict, Any, Optional, List, Union, TypedDict 
import re
from langchain_core.messages import HumanMessage, AIMessage 
from langchain_core.tools import tool 
from langgraph.checkpoint.memory import MemorySaver 
from langgraph.graph import END, START, StateGraph 
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

## 3. Cấu hình HuggingFace Token

Lấy HF token từ Kaggle secrets để tải model từ HuggingFace Hub.

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

## 4. Kiểm tra môi trường

Kiểm tra CUDA, GPU memory và PyTorch version trước khi load model.

In [4]:
# Kiểm tra môi trường CUDA
print("=== Environment Check ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch Version: {torch.__version__}")
print("=" * 50)

=== Environment Check ===
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 15.64 GB
PyTorch Version: 2.6.0+cu124


## 5. Load Base Model

Tải Mistral-7B-Instruct với cấu hình:
- **4-bit quantization** (NF4): Giảm memory footprint
- **BitsAndBytes**: Quantization backend
- **Device map auto**: Tự động phân bổ model lên GPU/CPU

In [5]:
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Loading tokenizer from {base_model_id}...")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    token=HF_TOKEN
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")

print(f"Loading base model from {base_model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

base_model.config.pad_token_id = tokenizer.pad_token_id

print("Base model loaded in 4-bit")

Loading tokenizer from mistralai/Mistral-7B-Instruct-v0.3...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Base model loaded in 4-bit


## 6. Load Fine-tuned Adapter & Merge

In [6]:
adapter_model_id = "/kaggle/input/datasets/linhtron123/finetune-mistral/final_model_best"  
if not os.path.exists(adapter_model_id):
    raise FileNotFoundError(f"Adapter path not found: {adapter_model_id}")

print(f"Found adapter at: {adapter_model_id}")

try:
    model = PeftModel.from_pretrained(base_model, adapter_model_id)
    print("PEFT adapter loaded successfully")
    
except Exception as e:
    print(f"Failed to load/merge PEFT adapter: {e}")
    raise

Found adapter at: /kaggle/input/datasets/linhtron123/finetune-mistral/final_model_best
PEFT adapter loaded successfully


## 7. Load Test Data

Load dataset chứa:
- **Dialogue**: Cuộc hội thoại giữa plaintiff và defendant
- **PLAINTIFF intent**: Ý định của nguyên đơn
- **DEFENDANT intent**: Ý định của bị đơn

In [7]:
def load_csv(file_path):
    return pd.read_csv(file_path)

csv_df = load_csv("/kaggle/input/test-intent-p2/test_intent.csv")
dialogues = csv_df["Dialogue"].tolist()
plaintiff_intents = csv_df["PLAINTIFF"].tolist()
defendant_intents = csv_df["DEFENDANT"].tolist()

## 8. Định nghĩa Manipulation Tactics

Danh sách 11 kỹ thuật manipulation được model nhận diện

In [8]:
allowed_tactics = [
    "persuasion", "playing the victim", "gaslighting", "evasion", 
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

## 9. Hàm Generation

Hàm sinh text từ model với:
- **torch.no_grad()**: Tắt gradient để tiết kiệm bộ nhớ
- **model.eval()**: Chế độ inference
- **Proper tokenization**: Pad token, EOS token
- **Temperature 0.4**: Cân bằng giữa creativity và consistency

In [9]:
def generate_response(prompt, max_new_tokens=1000):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    new_tokens = output[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


## 10. Agent State Definition

TypedDict định nghĩa state được chia sẻ giữa các agents:
- Input: dialogue, intents
- Intermediate: manipulation_detected, primary_manipulator, techniques
- Output: final_summary
- Messages: Lưu lịch sử conversation

In [10]:
class AgentState(TypedDict):
    dialogue: str
    dialogue_index: int
    plaintiff_intent: str
    defendant_intent: str
    manipulation_detected: Optional[str]
    primary_manipulator: Optional[str]
    techniques_used: Optional[List[str]]
    evidence: Optional[str]
    final_summary: Optional[Dict[str, Any]]
    messages: List[Union[HumanMessage, AIMessage]]

## 11. Detector Agent

**Agent đầu tiên** trong pipeline:
- **Nhiệm vụ**: Phát hiện có manipulation hay không
- **Output**: Yes/No
- **Parsing**: Regex robust để tìm Yes/No trong response
- **Routing**: Yes → Analyzer Agent, No → Meta Agent (skip analysis)

In [11]:
def detector_agent(state):
    dialogue = state["dialogue"]
    plaintiff_intent = state["plaintiff_intent"]
    defendant_intent = state["defendant_intent"]
    system_prompt = f"""<s>[INST] You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.

### Instructions:
1. Read the dialogue carefully.
2. Analyze it with the context of the intents provided
3. Determine if any manipulation is present or not.

Defendant's Intent: {defendant_intent}
Plaintiff's Intent: {plaintiff_intent}
Dialogue: {dialogue}

Question: Is there manipulation present in this dialogue?
You must respond with ONLY one word: Yes or No
[/INST]"""

    response = generate_response(system_prompt, max_new_tokens=5)  
    print(f"Raw LLM response: {response}")

    model_response = re.search(r"\[/INST\](.*?)$", response, re.DOTALL)
    if model_response:
        extracted_response = model_response.group(1).strip()
    else:
        extracted_response = response.strip()
    
    print(f"Full extracted response: {extracted_response}")
    lines = [line.strip() for line in extracted_response.split('\n') if line.strip()]
    if lines:
        last_line = lines[-1]
        print(f"Last non-empty line: {last_line}")
        
        if last_line.lower() == "yes":
            result = "Yes"
            print("Last line is exactly 'Yes'")
        elif last_line.lower() == "no":
            result = "No"
            print("Last line is exactly 'No'")
        else:
            result = "No"
            print("Last line is not exactly 'Yes' or 'No', defaulting to No")
    else:
        result = "No"
        print("No non-empty lines found, defaulting to No")
    
    print(f"Final result after parsing: {result}")
    
    return {
        **state,
        "manipulation_detected": result,
        "messages": state["messages"] + [AIMessage(content=f"Manipulation Present: {result}")]
    }

## 12. Analyzer Agent

**Agent thứ hai** (chỉ chạy nếu có manipulation):
- **Nhiệm vụ**: 
  - Xác định primary manipulator (Plaintiff/Defendant)
  - Nhận diện manipulation techniques được sử dụng
- **Input**: Dialogue + intents của cả hai bên
- **Output**: Primary manipulator + list of techniques

In [12]:
def analyzer_agent(state):   
    if state["manipulation_detected"] == "No":
        return {
            **state,
            "primary_manipulator": "None",
            "techniques_used": [ ],
            "messages": state["messages"] + [AIMessage(content="No manipulation detected, skipping analysis.")]
        }
    
    print("Manipulation detected, analyzing manipulator and techniques...")
    combined_prompt = f"""<s>[INST] Compare both dialogue and the given intents of both defendant and plaintiff, and analyze the manipulation:
    
Defendant's Intent: {state['defendant_intent']}
Plaintiff's Intent: {state['plaintiff_intent']}
Dialogue: {state['dialogue']}

First, determine who is the primary manipulator (answer with ONLY Defendant or Plaintiff).
Second, identify which manipulation techniques they used from this list:
{', '.join(allowed_tactics)}

Format your response exactly like this:
Primary Manipulator: [ ]
Techniques: [list ONLY the techniques that were actually used, separated by commas]
[/INST]"""

    print("Prompting model for manipulation analysis...")
    combined_response = generate_response(combined_prompt, max_new_tokens=100)
    print(f"Raw analyzer response: {combined_response}")
    primary_manipulator = "Unknown"
    if "primary manipulator: plaintiff" in combined_response.lower():
        primary_manipulator = "Plaintiff"
    elif "primary manipulator: defendant" in combined_response.lower():
        primary_manipulator = "Defendant"
    techniques_used = []
    for line in combined_response.split('\n'):
        if line.lower().startswith("techniques:"):
            techniques_part = line.split(':', 1)[1].strip()
            for technique in techniques_part.split(','):
                technique = technique.strip()
                for allowed_tactic in allowed_tactics:
                    if technique.lower() == allowed_tactic.lower():
                        techniques_used.append(allowed_tactic)
                        break
    
    analysis_result = f"Primary Manipulator: {primary_manipulator}\nManipulative Techniques: {', '.join(techniques_used) if techniques_used else 'None'}"
    
    print(f"ANALYZER AGENT RESULT:\n{analysis_result}")
    
    return {
        **state,
        "primary_manipulator": primary_manipulator,
        "techniques_used": techniques_used,
        "messages": state["messages"] + [AIMessage(content=analysis_result)]
    }

## 13. Evidence Agent

**Agent thứ ba** - Validation layer:
- **Nhiệm vụ**: Xác thực kết quả từ Analyzer Agent với bằng chứng cụ thể
- **Cross-check**: So sánh manipulation techniques với dialogue thực tế
- **Refinement**: Loại bỏ techniques không có bằng chứng rõ ràng
- **Output**: Confirmed manipulator + confirmed techniques

In [13]:
def evidence_agent(state):
    dialogue = state["dialogue"]
    primary_manipulator = state["primary_manipulator"]
    techniques_used = state["techniques_used"]
    prompt = f"""<s>[INST] Analyze this dialogue for manipulation.

Dialogue: {dialogue}
Current analysis:
- Primary Manipulator: {primary_manipulator}
- Techniques identified: {', '.join(techniques_used) if techniques_used else 'None'}

For the current analysis, validate if the primary manipulator is correct and which techniques are actually used with clear evidence.

Format your response EXACTLY like this:
Confirmed Manipulator: [ ]
Confirmed Techniques: [ ]

Only include techniques from this list with clear evidence: {', '.join(allowed_tactics)}
[/INST]"""

    print("Prompting model for evidence validation...")
    validation_response = generate_response(prompt, max_new_tokens=500)
    print(f"Raw evidence response: {validation_response}")
    
    confirmed_manipulator = primary_manipulator  
    confirmed_techniques = []
    
    if "confirmed manipulator: plaintiff" in validation_response.lower():
        confirmed_manipulator = "Plaintiff"
    elif "confirmed manipulator: defendant" in validation_response.lower():
        confirmed_manipulator = "Defendant"
    
    for line in validation_response.split('\n'):
        if line.lower().startswith("confirmed techniques:"):
            techniques_part = line.split(':', 1)[1].strip()
            techniques_part = techniques_part.strip('[]')
            
            if techniques_part.lower() != "none":
                for technique in techniques_part.split(','):
                    technique = technique.strip()
                    for allowed_tactic in allowed_tactics:
                        if technique.lower() == allowed_tactic.lower():
                            confirmed_techniques.append(allowed_tactic)
                            break
    if not confirmed_techniques:
        for tactic in allowed_tactics:
            pattern = r'[0-9.•\-\*]+\s*' + re.escape(tactic) + r'[:\s]'
            if re.search(pattern, validation_response, re.IGNORECASE):
                confirmed_techniques.append(tactic)
    techniques_string = ', '.join(confirmed_techniques) if confirmed_techniques else 'None'
    
    validation_result = f"""Evidence Assessment:
Confirmed Manipulator: {confirmed_manipulator}
Confirmed Techniques: {techniques_string}
"""
    
    print(f"EVIDENCE AGENT RESULT:\nManipulator: {confirmed_manipulator}\nTechniques: {confirmed_techniques}")
    
    return {
        **state,
        "primary_manipulator": confirmed_manipulator,
        "techniques_used": confirmed_techniques,
        "evidence": validation_result,
        "messages": state["messages"] + [AIMessage(content=validation_result)]
    }

## 14. Meta Agent

**Agent cuối cùng** - Aggregation & Decision:
- **Nhiệm vụ**: Tổng hợp kết quả từ tất cả agents
- **Logic**: 
  - Nếu có manipulation + manipulator + techniques → Yes
  - Ngược lại → No
- **Output**: Final summary với 3 fields:
  - manipulation_present: Yes/No
  - primary_manipulator: Plaintiff/Defendant/None
  - techniques: List of confirmed techniques

In [14]:
def meta_agent(state):
    
    manipulation_present = state["manipulation_detected"]
    primary_manipulator = state["primary_manipulator"]
    techniques = state["techniques_used"]
    
    print(f"Preparing final summary from:\nManipulation: {manipulation_present}\nManipulator: {primary_manipulator}\nTechniques: {techniques}")
    if manipulation_present == "Yes" and primary_manipulator not in ["None", "Unknown"] and techniques:
        final_manipulation = "Yes"
        final_manipulator = primary_manipulator
        final_techniques = techniques
    else:
        if manipulation_present == "Yes" and primary_manipulator not in ["None", "Unknown"] and not techniques:
            analyzer_detected_techniques = state.get("techniques_used", [])
            if analyzer_detected_techniques:
                final_manipulation = "Yes"
                final_manipulator = primary_manipulator
                final_techniques = analyzer_detected_techniques
            else:
                final_manipulation = "No"
                final_manipulator = "None"
                final_techniques = []
        else:
            final_manipulation = "No"
            final_manipulator = "None"
            final_techniques = []
    
    formatted_response = f"""Manipulation Present: {final_manipulation}
Primary Manipulator: {final_manipulator}
Manipulative Techniques: {", ".join(final_techniques) if final_techniques else "None"}"""

    final_result = {
        "manipulation_present": final_manipulation,
        "primary_manipulator": final_manipulator,
        "techniques": final_techniques
    }
    
    print(f"META AGENT FINAL OUTPUT:\n{formatted_response}")
    
    return {
        **state,
        "final_summary": final_result,
        "messages": state["messages"] + [AIMessage(content=formatted_response)]
    }

## 15. Workflow Graph & Processing Functions

### LangGraph Workflow:
- **StateGraph**: Định nghĩa flow giữa các agents
- **Conditional edges**: Routing logic (Yes/No branching)
- **Entry point**: Detector Agent
- **End point**: Meta Agent

### Processing Functions:
- `process_dialogue()`: Xử lý 1 dialogue qua toàn bộ pipeline
- `save_results_to_csv()`: Xử lý tất cả dialogues và lưu kết quả

In [15]:
workflow_graph = StateGraph(AgentState)
workflow_graph.add_node("detector_agent", detector_agent)
workflow_graph.add_node("analyzer_agent", analyzer_agent)
workflow_graph.add_node("evidence_agent", evidence_agent)
workflow_graph.add_node("meta_agent",     meta_agent)

workflow_graph.set_entry_point("detector_agent")

workflow_graph.add_conditional_edges(
    "detector_agent",
    lambda state: "analyzer_agent" if state["manipulation_detected"] == "Yes" else "meta_agent"
)
workflow_graph.add_edge("analyzer_agent", "evidence_agent")
workflow_graph.add_edge("evidence_agent", "meta_agent")
workflow_graph.add_edge("meta_agent",     END)

workflow = workflow_graph.compile()

In [16]:
def process_dialogue(dialogue_idx):
    initial_state = AgentState({
        "dialogue":              dialogues[dialogue_idx],
        "dialogue_index":        dialogue_idx,
        "plaintiff_intent":      plaintiff_intents[dialogue_idx],
        "defendant_intent":      defendant_intents[dialogue_idx],
        "manipulation_detected": None,
        "primary_manipulator":   None,
        "techniques_used":       None,
        "evidence":              None,
        "final_summary":         None,
        "messages": [HumanMessage(content=f"Analyze dialogue {dialogue_idx}")]
    })
    return workflow.invoke(initial_state)

In [17]:
def save_results_to_csv():
    manipulation_results = []
    manipulator_results  = []
    techniques_results   = []

    for idx in range(len(dialogues)):
        print(f"\n=== Dialogue {idx}/{len(dialogues)-1} ===")
        try:
            result        = process_dialogue(idx)
            summary       = result.get("final_summary", {})
            manip_present = summary.get("manipulation_present", "No")
            manip_person  = summary.get("primary_manipulator",  "None")
            techs         = summary.get("techniques", [])

        except Exception as e:
            import traceback
            print(f"  [ERROR] Dialogue {idx}:")
            traceback.print_exc()
            manip_present = "Error"
            manip_person  = "Error"
            techs         = []

        manipulation_results.append(manip_present)
        manipulator_results.append(manip_person)
        techniques_results.append(", ".join(techs) if techs else "None")

        print(f"  → {manip_present} | {manip_person} | {', '.join(techs) if techs else 'None'}")

        if idx % 20 == 0 and idx > 0:
            pd.DataFrame({
                "Dialogue_Index":       list(range(idx + 1)),
                "Dialogue":             dialogues[:idx + 1],
                "Manipulation_Present": manipulation_results,
                "Primary_Manipulator":  manipulator_results,
                "Techniques_Used":      techniques_results,
            }).to_csv("/kaggle/working/checkpoint_claim.csv", index=False)
            print(f"  [Checkpoint saved at idx={idx}]")

    output_df = csv_df[["Dialogue", "PLAINTIFF", "DEFENDANT"]].copy()
    output_df.insert(0, "Dialogue_Index", range(len(output_df)))
    output_df["Manipulation_Present"] = manipulation_results
    output_df["Primary_Manipulator"]  = manipulator_results
    output_df["Techniques_Used"]      = techniques_results

    output_path = "/kaggle/working/claim_results.csv"
    output_df.to_csv(output_path, index=False)
    print(f"\n[DONE] Results saved to: {output_path}")
    return output_path

## 16. Run Full Pipeline

In [18]:
if __name__ == "__main__":
    save_results_to_csv()


=== Dialogue 0/154 ===
Raw LLM response: No
Full extracted response: No
Last non-empty line: No
Last line is exactly 'No'
Final result after parsing: No
Preparing final summary from:
Manipulation: No
Manipulator: None
Techniques: None
META AGENT FINAL OUTPUT:
Manipulation Present: No
Primary Manipulator: None
Manipulative Techniques: None
  → No | None | None

=== Dialogue 1/154 ===
Raw LLM response: Yes
Full extracted response: Yes
Last non-empty line: Yes
Last line is exactly 'Yes'
Final result after parsing: Yes
Manipulation detected, analyzing manipulator and techniques...
Prompting model for manipulation analysis...
Raw analyzer response: Primary Manipulator: Plaintiff
Techniques: playing the victim, framing the narrative
ANALYZER AGENT RESULT:
Primary Manipulator: Plaintiff
Manipulative Techniques: playing the victim, framing the narrative
Prompting model for evidence validation...
Raw evidence response: Confirmed Manipulator: Plaintiff
Confirmed Techniques: playing the victim, 

## 17. Evaluation

In [19]:
df_results = pd.read_csv("/kaggle/working/claim_results.csv")
df_test = pd.read_csv('/kaggle/input/datasets/linhtron123/legaldata/test_split.csv')  

test = pd.merge(df_results, df_test, on='Dialogue', how='left')
print(test.head())

   Dialogue_Index                                           Dialogue  \
0               0  Judge: I understand, but my hypothetical assum...   
1               1  Judge: Well, but if -- why -- your voluntary d...   
2               2  Judge: Why did you go?\nPlaintiff: I had nothi...   
3               3  Judge: Tell me what happened.\nPlaintiff: Okay...   
4               4  Judge: Is your position— is there any daylight...   

                                           PLAINTIFF  \
0  The Plaintiff's (Lawyer of Defendant) statemen...   
1  The Plaintiff's goal or motive behind their wo...   
2  The Plaintiff's statements suggest that their ...   
3  The Plaintiff's statements suggest that they h...   
4  The Plaintiff's statements suggest that they a...   

                                           DEFENDANT Manipulation_Present  \
0  The Defendant's lawyer is arguing that the Def...                   No   
1  The Defendant's lawyer is arguing that the Def...                  Yes   

### Task 1: Manipulation Detection (Binary Classification)¶
Đánh giá khả năng phát hiện có/không có manipulatio

In [20]:
print("True label distribution:\n", test["Manipulative"].value_counts(dropna=False))
print("\nPredicted label distribution:\n", test["Manipulation_Present"].value_counts(dropna=False))

True label distribution:
 Manipulative
1    95
0    60
Name: count, dtype: int64

Predicted label distribution:
 Manipulation_Present
Yes    105
No      50
Name: count, dtype: int64


In [21]:
test["Manipulation_Present"] = test["Manipulation_Present"].map({'Yes': 1, 'No': 0})
targets = test["Manipulative"].astype(int).tolist()
manipulative_preds = test["Manipulation_Present"].astype(int).tolist()

In [22]:
accuracy = accuracy_score(targets, manipulative_preds)
precision = precision_score(targets, manipulative_preds, zero_division=0)
recall = recall_score(targets, manipulative_preds, zero_division=0)
f1 = f1_score(targets, manipulative_preds, average="binary", zero_division=0)
conf_matrix = confusion_matrix(targets, manipulative_preds)

print("="*50)
print("TASK 1: MANIPULATION DETECTION")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"              No      Yes")
print(f"Actual No   {conf_matrix[0][0]:4d}    {conf_matrix[0][1]:4d}")
print(f"Actual Yes  {conf_matrix[1][0]:4d}    {conf_matrix[1][1]:4d}")

TASK 1: MANIPULATION DETECTION
Accuracy:  0.8710
Precision: 0.8571
Recall:    0.9474
F1 Score:  0.9000

Confusion Matrix:
                Predicted
              No      Yes
Actual No     45      15
Actual Yes     5      90


### Task 2: Primary Manipulator Identification
Đánh giá khả năng xác định manipulator (Plaintiff/Defendant/None).

In [23]:
test["Primary_Manipulator"] = test["Primary_Manipulator"].fillna("None").str.strip()
test["Primary Manipulator"] = test["Primary Manipulator"].fillna("None").str.strip()

print("Ground Truth unique values:", test["Primary_Manipulator"].unique())
print("Predicted unique values:", test["Primary Manipulator"].unique())

all_categories = list(set(
    test["Primary_Manipulator"].tolist() + test["Primary Manipulator"].tolist()
))
encoder = LabelEncoder()
encoder.fit(all_categories)

targets_encoded = encoder.transform(test["Primary_Manipulator"])
preds_encoded = encoder.transform(test["Primary Manipulator"])

label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
print("\nLabel Encoding Mapping:", label_mapping)

Ground Truth unique values: ['None' 'Plaintiff' 'Defendant']
Predicted unique values: ['None' 'Plaintiff' 'Defendant']

Label Encoding Mapping: {'Defendant': 0, 'None': 1, 'Plaintiff': 2}


In [24]:
accuracy = accuracy_score(targets_encoded, preds_encoded)
precision = precision_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
recall = recall_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
f1 = f1_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
conf_matrix = confusion_matrix(targets_encoded, preds_encoded)

print("\n" + "="*50)
print("TASK 2: PRIMARY MANIPULATOR IDENTIFICATION")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f} (weighted)")
print(f"Recall:    {recall:.4f} (weighted)")
print(f"F1 Score:  {f1:.4f} (weighted)")
print(f"\nConfusion Matrix:")
print(conf_matrix)
print(f"\nLabel mapping: {dict(zip(encoder.transform(encoder.classes_), encoder.classes_))}")


TASK 2: PRIMARY MANIPULATOR IDENTIFICATION
Accuracy:  0.7355
Precision: 0.7332 (weighted)
Recall:    0.7355 (weighted)
F1 Score:  0.7260 (weighted)

Confusion Matrix:
[[46  8  5]
 [ 1 45  4]
 [16  7 23]]

Label mapping: {0: 'Defendant', 1: 'None', 2: 'Plaintiff'}


### Task 3: Techniques Manipulator Identification
Đánh giá khả năng xác định các kỹ thuật manipulator (11 kỹ thuật).

In [25]:
test["Manipulation Techniques"] = test["Manipulation Techniques"].fillna("None").astype(str)
test["Techniques_Used"] = test["Techniques_Used"].fillna("None").astype(str)

print("Sample Ground Truth techniques:")
print(test["Manipulation Techniques"].head(10))
print("\nSample Predicted techniques:")
print(test["Techniques_Used"].head(10))

def parse_techniques(x):
    if pd.isna(x) or x == "None" or x == "nan":
        return ["None"]
    return [t.strip().lower() for t in str(x).split(",") if t.strip()]

test["Manipulation Techniques"] = test["Manipulation Techniques"].apply(parse_techniques)
test["Techniques_Used"] = test["Techniques_Used"].apply(parse_techniques)

# Kiểm tra unique labels
unique_gt_labels = set(label for labels in test["Manipulation Techniques"] for label in labels)
unique_pred_labels = set(label for labels in test["Techniques_Used"] for label in labels)

# Multi-label binarization
mlb = MultiLabelBinarizer()
all_labels = list(unique_gt_labels | unique_pred_labels)
mlb.fit([all_labels])

y_true = mlb.transform(test["Manipulation Techniques"])
y_pred = mlb.transform(test["Techniques_Used"])

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
jaccard = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

print("\n" + "="*50)
print("TASK 3: MANIPULATION TECHNIQUES CLASSIFICATION")
print("="*50)
print(f"Accuracy:      {accuracy:.4f} (exact match)")
print(f"Precision:     {precision:.4f} (macro)")
print(f"Recall:        {recall:.4f} (macro)")
print(f"F1 Score:      {f1:.4f} (macro)")
print(f"Jaccard Score: {jaccard:.4f} (samples)")

Sample Ground Truth techniques:
0                                                 None
1                                                 None
2                             deflection, minimization
3    gaslighting, deflection, emotional appeal, pla...
4                                                 None
5                       deflection, playing the victim
6                                                 None
7    gaslighting, character attack, playing the victim
8                                                 None
9          gaslighting, deflection, playing the victim
Name: Manipulation Techniques, dtype: object

Sample Predicted techniques:
0                                                 None
1            playing the victim, framing the narrative
2              emotional appeal, framing the narrative
3                         deflection, emotional appeal
4                                                 None
5           deflection, minimization, emotional appeal
6            

### Final Summary
Tổng hợp tất cả metrics từ 3 tasks.

In [26]:
print("\n" + "="*60)
print(" "*15 + "FINAL EVALUATION SUMMARY")
print("="*60)
print("\nDataset Size:", len(test))
print("\n" + "-"*60)
print(f"{'Task':<45} {'Metric':<10} {'Score':<10}")
print("-"*60)

# Task 1
task1_acc = accuracy_score(targets, manipulative_preds)
task1_f1 = f1_score(targets, manipulative_preds, average="binary", zero_division=0)
print(f"{'1. Manipulation Detection (Binary)':<45} {'Accuracy':<10} {task1_acc:.4f}")
print(f"{'':<45} {'F1':<10} {task1_f1:.4f}")

# Task 2  
task2_acc = accuracy_score(targets_encoded, preds_encoded)
task2_f1 = f1_score(targets_encoded, preds_encoded, average="weighted", zero_division=0)
print(f"{'2. Primary Manipulator (3-class)':<45} {'Accuracy':<10} {task2_acc:.4f}")
print(f"{'':<45} {'F1 (wtd)':<10} {task2_f1:.4f}")

# Task 3
task3_acc = accuracy_score(y_true, y_pred)
task3_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
task3_jaccard = jaccard_score(y_true, y_pred, average="samples", zero_division=0)
print(f"{'3. Techniques Classification (Multi-label)':<45} {'Accuracy':<10} {task3_acc:.4f}")
print(f"{'':<45} {'F1 (macro)':<10} {task3_f1:.4f}")
print(f"{'':<45} {'Jaccard':<10} {task3_jaccard:.4f}")

print("="*60)
print("\nEvaluation completed!")


               FINAL EVALUATION SUMMARY

Dataset Size: 155

------------------------------------------------------------
Task                                          Metric     Score     
------------------------------------------------------------
1. Manipulation Detection (Binary)            Accuracy   0.8710
                                              F1         0.9000
2. Primary Manipulator (3-class)              Accuracy   0.7355
                                              F1 (wtd)   0.7260
3. Techniques Classification (Multi-label)    Accuracy   0.2903
                                              F1 (macro) 0.2107
                                              Jaccard    0.3942

Evaluation completed!
